# Intake: Ahneman

The Ahneman *et al.* (Science 2018) Buchwald-Hartwig C-N coupling screen --
4 ligands x 3 bases x 15 aryl halides x 22 additives, 3,955 reactions after
the exclusions below.

Source: [doylelab/rxnpredict](https://github.com/doylelab/rxnpredict)

This one rebuilds **from the raw download**, which is cached in `raw/`.
`rxnprep.py` beside this notebook holds the Ahneman-specific reshaping; the
bundle writing is the shared `gpc.prep` code.

Only 4 ligands, which is worth remembering when reading its results: LOLO is
4 folds, each removing a quarter of the data, and its matched control is
`iid_stratified_4`. The screen's size comes from varying everything *else*.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()            # datasets/ahneman/
ROOT = HERE.parent.parent              # gp_collab_hub/
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(HERE))

from gpc import prep
from gpc.data import load_bundle

pd.set_option("display.width", 220, "display.max_columns", 80)

OUT = HERE / "inputs"                  # the bundle this notebook writes

import rxnprep as R

# ---- Editable -----------------------------------------------------------
DATASET      = "ahneman"
DISPLAY_NAME = "Ahneman"
TARGET       = "yield"
GROUP        = "ligand"
CATEGORICAL  = ["aryl_halide", "base", "additive"]
SOURCE_URL   = "https://github.com/doylelab/rxnpredict"

RAW     = HERE / "raw"      # download cache; already populated
REFRESH = False             # True re-downloads from GitHub
# --------------------------------------------------------------------------

sources = R.download_sources(RAW, refresh=REFRESH)
display(pd.DataFrame(sources).T[["bytes", "sha256"]])

## 1. Is the descriptor join correct?

`raw/output_table.csv` has no key columns and the original R script attaches
yields by row position, which is fragile. Instead the full factorial grid is
rebuilt from the four per-component tables and compared against the published
table. `identical_row_multiset: True` means they agree exactly, ignoring order.

In [ ]:
check = R.verify_against_doyle(RAW)
display(pd.Series(check).to_frame("result"))
assert check["same_columns"] and check["identical_row_multiset"], \
    "join does not reproduce the published table"

## 2. Build the reactions table

Reaction conditions and yield, nothing else. Two groups of rows are dropped,
both reported in the audit:

* **no-aryl-halide control wells**, blank by design (every one is 0% yield);
* **one additive with no computed descriptors** (`5-Phenyl-1,2,4-oxadiazole`),
  plus wells with no additive.

What remains is 3,955 of a 3,960-cell grid -- five combinations were never run.

In [ ]:
reactions, audit = R.build_reactions(RAW)
audit["sources"] = sources

print(f"{reactions.shape[0]} reactions x {reactions.shape[1]} columns")
display(pd.DataFrame([
    {"reason": "no-aryl-halide controls",
     "rows": audit["dropped_no_aryl_halide_controls"]["rows"],
     "note": f"max yield {audit['dropped_no_aryl_halide_controls']['max_yield']}"},
    {"reason": "additive without descriptors",
     "rows": audit["dropped_additive_without_descriptors"]["rows"],
     "note": ", ".join(audit["dropped_additive_without_descriptors"]["names"])},
]))
print(f"kept {audit['kept_rows']} of a {audit['full_factorial']}-cell grid")
display(reactions.head(3))

## 3. The ligand mapping

All four ligands are in the Kraken reference. `verify_ligand_map` prints both
SMILES side by side -- check they agree chemically before trusting the mapping,
since the two sources write them differently.

In [ ]:
# ---- Editable -----------------------------------------------------------
MAPPING = {"XPhos": 1, "t-BuXPhos": 90, "t-BuBrettPhos": 89, "AdBrettPhos": 347}
# --------------------------------------------------------------------------

display(R.verify_ligand_map(RAW))
reactions = prep.attach_kraken(reactions.drop(columns=["kraken_id"], errors="ignore"),
                               GROUP, MAPPING)

## 4. Describe it, then validate

The table below is the record of what this bundle claims about itself: rows and
target statistics per group. Read it before writing -- an unbalanced design
shows up here, and it changes how the LOLO folds should be read (their sizes
follow these counts).

In [ ]:
display(prep.describe(reactions, {"data": {"target": TARGET, "group": GROUP}}))

In [ ]:
# ---- Editable: what this dataset IS -------------------------------------
cfg = prep.bundle_config(
    dataset=DATASET,
    display_name=DISPLAY_NAME,
    target=TARGET,
    group=GROUP,
    categorical=CATEGORICAL,
    reactions=reactions,
    source=SOURCE_URL,
    # models/methods default sensibly: all five sections, and the method list
    # whose matched control has this screen's own group count. Pass explicit
    # lists here to override.
)
# --------------------------------------------------------------------------

display(pd.json_normalize(cfg["data"]).T.rename(columns={0: "value"}))
print("methods:", cfg["evaluation"]["methods"])

# Every check load_bundle will make, run here so a bad bundle fails at the
# point it was built rather than on the cluster three hours into a job.
display(prep.check_bundle(reactions, cfg, prep.kraken_features()))

## 5. Write the bundle

`write_bundle` writes `reactions.csv`, the Kraken descriptor rows for the
ligands this screen uses, `ligand_mapping.csv`, the copied reference PCA and
`config.json`. The PCA is **copied, never refitted** -- PC1..PC4 have to mean
the same axes in every dataset or `pc_top` and `pc_scores` stop being
comparable, which is the point of running them.

In [ ]:
bundle = prep.write_bundle(OUT, reactions, cfg, MAPPING, audit=audit)

# Read it straight back through the engine's own loader: if this succeeds, the
# bundle is usable by `gpc train` exactly as written.
back, ligands, reference, prepared = load_bundle(bundle)
print(f"\nreloaded: {len(back)} rows, {back[cfg['data']['group']].nunique()} "
      f"{cfg['data']['group']}s, {len(ligands)} descriptor rows, "
      f"PCA over {len(reference.columns)} descriptors")
display(pd.DataFrame([{"file": p.name, "KB": round(p.stat().st_size / 1024, 1)}
                      for p in sorted(bundle.iterdir())]))
print("\nNext: build_run.ipynb in the hub root, and pick this dataset.")